**Author:** Amanda Baright

**Date:** 04.28.2026

**Purpose:** ST 554 Final Project

# ST 554 Final Project

This final project will assess my ability to use spark to handle streaming data, use spark for fitting a machine learning model, and a few other things. The project will be split into three main components: Fitting a machine learning model using `pyspark`'s `MLlib` module, reading in a stream of data that is produced ourselves using a `.py` file, and using the model to do predictions on the stream and write the predictions out to the console.

The data is modified from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), where the study was about relating power consumption from different time zones of Tetouan city to various factors like time of day, temperatrue, and humidity. We will treat the `Power_Zone_3` variable as our response variable, and will use all of the other varialbes as predictors. This will allow us to predict the value of `Power_Zone_3` appropriately if that reading goes offline in the future.

We are provided a chunk of the modified data to build our model on, and will then "stream data" to a folder in the form of CSV files. We will monitor this folder, and will use the fitted model to predict on the incoming data.

## Fitting Your Model

This first part will focus on model fitting. We will use the `power_ml_data.csv` which is available at: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. We first want to read in this data using a standard pandas data frame using `pd.read_csv()`. We will also use this first step to read in all of the libraries we may need. We will also initialize a Spark session and convert this pandas data frame into a spark data frame.

In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (SQLTransformer, Binarizer, OneHotEncoder, 
                                VectorAssembler, PCA, StandardScaler)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder.getOrCreate()

# Read data into pandas
pandas_df = pd.read_csv("power_ml_data.csv")

# Convert to spark data frame
power_df = spark.createDataFrame(pandas_df)

# Look at first few rows
power_df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 22:16:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 22:16:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We then want to start building a `MLlib` Pipeline that includes all of the pre-specified transformations.

The first step will be to use a SQL transformer to rename `Power_Zone_3` as `label` and cast the `Hour` column to be stored as a `DoubleType` variable.

In [4]:
sqlTrans = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label, CAST(Hour AS DOUBLE) AS HourDouble FROM __THIS__"
)

We then want to add a component that binarizes the `Hour` column based on the column being less than 6.5 or not, which essentially breaks the `Hour` column into night vs day.

In [5]:
binarizer = Binarizer(threshold=6.5, inputCol="HourDouble", outputCol="binHour")

We then want to do one-hot encoding for the `Month` column, where each unique month in the `Month` column is transformed into its own binary column with 1 indicating the presence of that month category and 0 indicates the absence.

In [6]:
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["Month_vec"])

We now want to use `VectorAssembler()` to assemble the weather related variables (`Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`). Additionally, we will want to standardize the weather variables before doing the PCA fit, as PCA is sensitive to the scale of the data.

In [7]:
weather_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="weather_features"
)

scaler_pca = StandardScaler(inputCol="weather_features", outputCol="scaled_weather", 
                            withStd=True, withMean=True)

Now that we standardized our weather features, we can run a PCA fit using two PCs in the transformation.

In [8]:
pca = PCA(k=2, inputCol="scaled_weather", outputCol="pca_features")

We then do a final feature assembly where we include the `pca_features` that were selected with the PCA fit, the binary `Hour` variable, `Power_Zone_1`, `Power_Zone_2`, and the `Month` indicator variable. These will be stored into a `unscaled_features` column by using the `VectorAssembler()` method again. We then want to standardize the final features for the Elastic Net Model, as the Elastic Net model adds a penalty term to the magnitude of the coefficients so features with larger scales will have unfairly small coefficients which may lead to some predictors to be ignored.

In [9]:
final_assembler = VectorAssembler(
    inputCols=["pca_features", "binHour", "Power_Zone_1", "Power_Zone_2", "Month_vec"],
    outputCol="unscaled_features"
)

scaler_final = StandardScaler(inputCol="unscaled_features", outputCol="features", 
                              withStd=True, withMean=True)

Now we can finally define the pipeline with all of these amazing transformations!

In [10]:
pipeline = Pipeline(stages=[
    sqlTrans, 
    binarizer, 
    encoder, 
    weather_assembler, 
    scaler_pca, 
    pca, 
    final_assembler, 
    scaler_final
])

We then want to use the `CrossValidator()` function with the `LinearRegression()` function to fit an elastic net model. We will use a pretty extensive grid with the `ParamGridBuilder()` function to examine all combinations of a few parameters for `regParam` and `elasticNetParam`. We will also fit the model using 5-fold CR with Root Mean Square Error (`rmse`) as our criterion.

In [11]:
# Set up the LinearRegression() function
lr = LinearRegression(featuresCol="features", labelCol="label")

# Define the extensive grid selection
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Set up 5-fold CV with rmse as our metric of choice.
cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=RegressionEvaluator(metricName="rmse"),
                    numFolds=5)

Now we can finally fit the entire workflow with all of our components built above. The first step is to fit the pipeline to the data.

In [12]:
pipelineModel = pipeline.fit(power_df)

Next we will transform our data using our pipeline to get the `features` and `label` columns.

In [13]:
transformedData = pipelineModel.transform(power_df)

We can now fit the `CrossValidator()` on the transformed data.

In [14]:
cvModel = cv.fit(transformedData)

26/04/28 22:17:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 22:17:23 WARN Instrumentation: [2b9efc40] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 22:17:25 WARN Instrumentation: [2b9efc40] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 22:17:26 WARN Instrumentation: [cd06af9d] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 22:17:26 WARN Instrumentation: [cd06af9d] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 22:17:27 WARN Instrumentation: [46b5d563] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 22:17:27 WARN Instrumentation: [46b5d563] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

We then want to look at the best model and report the optimal values chosen for the tuning parameters and the CV error. Here we can extract the best model with the `.bestModel` attribute. We can then print out the `regParam` and `elasticNetParam` with their respective `.getParam()` method and find the average RMSE from the Cross Validation run to get the CV error.

In [15]:
bestModel = cvModel.bestModel
print(f"Optimal regParam: {bestModel.getRegParam()}")
print(f"Optimal elasticNetParam: {bestModel.getElasticNetParam()}")
print(f"CV RMSE: {min(cvModel.avgMetrics)}")

Optimal regParam: 0.05
Optimal elasticNetParam: 0.25
CV RMSE: 2175.1994930451115


We now want to report the training RMSE using a similar method as done in the ST 554 course notes. This includes using the fitted model as a transformer and evaluating on the entire training set. Here we will find the predictions using the `.transform()` method on the transformed data. We can then use the `RegressionEvaluator()` function to evaluate the fit using RMSE as the metric.

In [16]:
predictions = cvModel.transform(transformedData)
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
print(f"Training RMSE: {evaluator.evaluate(predictions)}")

Training RMSE: 2174.4489366667844


We then want to take the outputted transformations from the model (i.e. the predictions) and create a `residual` column. A residual is the observation (`label`) minus the prediction (`prediction`). We can utilize the `.withColumn()` method to help create this new `residual` column. We then will print out the data frame with these `residual`s, the `label` column, and the `predictions`. Here we can use `.select()` with `.show()` to print out these columns.

In [17]:
predictions = predictions.withColumn("residual", predictions.label - predictions.prediction)
predictions.select("label", "prediction", "residual").show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20935.38430438479|-694.4204443847884|
|20131.08434|18701.555910544877|1429.5284294551238|
|19668.43373| 18237.19022874543| 1431.243501254572|
|18899.27711| 17615.91633473588|1283.3607752641183|
|18442.40964|17017.446116700812|1424.9635232991895|
|18130.12048|16540.704515424233|1589.4159645757682|
|17945.06024|16114.793519477189|  1830.26672052281|
|17459.27711|15742.169383359811| 1717.107726640188|
|17025.54217| 15282.71283241262|  1742.82933758738|
|16794.21687|14935.073016697635|1859.1438533023647|
|16638.07229|14645.415619662383|1992.6566703376175|
|16395.18072|14395.849255080117|1999.3314649198837|
|16117.59036|14073.973811005182| 2043.616548994818|
| 15822.6506|13609.485062276153|2213.1655377238476|
|15672.28916|13432.410860275868|2239.8782997241324|
|15597.10843|13282.506202886965| 2314.602227113035|
|15510.36145

## Streaming Part

We now want to start the streaming part of the final project. Here we have another file `power_streaming_data.csv` that is available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv. We will want to ensure it's stored where our `.py` file we create later will find it. We will randomly sample rows from this file to output to `.csv` files that we will be reading in.

### Reading a Stream

We're going to read in a stream in the form of `.csv` files. We will want a folder where we will be sending these created `.csv` files. Here I created a folder called `streaming_folder` for convenience sake. However, our first step for this step will be setting up the schema for the stream, where we will just use the schema from the original data for simplicity sake.

In [18]:
from pyspark.sql.types import StructType, DoubleType, IntegerType
import pyspark.sql.functions as F

schema = StructType() \
    .add("Temperature", DoubleType()) \
    .add("Humidity", DoubleType()) \
    .add("Wind_Speed", DoubleType()) \
    .add("General_Diffuse_Flows", DoubleType()) \
    .add("Diffuse_Flows", DoubleType()) \
    .add("Power_Zone_1", DoubleType()) \
    .add("Power_Zone_2", DoubleType()) \
    .add("Power_Zone_3", DoubleType()) \
    .add("Month", IntegerType()) \
    .add("Hour", IntegerType())

We then want to set up the `readStream`, making sure we include the `header = True` as we will output files with a header and don't need to read that in.

In [19]:
streaming_df = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .csv("streaming_folder")

### Transform / Aggregation Step

Now we will do two separate things on the stream and join them together!

With the stream, we will use the model transformer to obtain predictions from the incoming data. On the resulting predicitons we will also create a `residual` column as done in the previous section where we will only return the `label`, `prediction`, and `residual` column.

In [20]:
# We use the fitted models from the "Fitting Your Model" section to get the predictions
pipeline_transformed = pipelineModel.transform(streaming_df)
predictions_df = cvModel.transform(pipeline_transformed)

stream_1 = predictions_df.withColumn("residual", F.col("label") - F.col("prediction")) \
    .select("label", "prediction", "residual")

We then want to use another transformation on the (original) stream, to modify the response variable to be called `label` to help facilitate the join. That is, we will use the `streaming_df` and modify `Power_Zone_3` to be called `label` by using `withColumnRenamed()

In [21]:
stream_2 = streaming_df.withColumnRenamed("Power_Zone_3", "label") \
    .select("label")

We then want to join the above transformations with an inner join based on the `label` variable, as it is common to both streams.

In [22]:
final_stream = stream_1.join(stream_2, "label")

### Writing Step

We now want to write the stream to the console using the `append` output mode and start the query!

In [23]:
query = final_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/28 22:28:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d79f47c5-fd89-49a0-9cf5-27d3f54cbca4. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/28 22:28:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [24]:
query.stop()

26/04/28 22:28:44 WARN DAGScheduler: Failed to cancel job group 470a8fb9-03ac-47e5-a44b-ad5814076a3e. Cannot find active jobs for it.
26/04/28 22:28:44 WARN DAGScheduler: Failed to cancel job group 470a8fb9-03ac-47e5-a44b-ad5814076a3e. Cannot find active jobs for it.
